# مرحلهٔ دوم: پشت میز DBA (Olist روی PostgreSQL)

این نوت‌بوک برای هر پنج سؤال تیم محصول، نسخهٔ اولیهٔ کوئری را کنار نسخهٔ بهینه اجرا می‌کند، پلن اجرای هر دو را نشان می‌دهد و هر ادعای گزارش را با یک فایل شاهد در `data/curated/evidence/` پشتیبانی می‌کند. منطق اندازه‌گیری در پکیج `src/olist_perf` است و نوت‌بوک فقط آن را صدا می‌زند و روایت می‌کند.

### قرارداد اجرا

1. پایگاه‌داده با **یک فرمان** از PostgreSQL خالی ساخته و پر می‌شود (از پوشهٔ مرحلهٔ اول):
   `python -m data_factory load-database --config config/default.toml --reset --skip-factory-sink`
   این فرمان مخرب است (schema `olist` را از نو می‌سازد)، پس نوت‌بوک آن را اجرا نمی‌کند. زمان این بارگذاری جزو سقف ۱۰ دقیقه نیست.
2. رمز پایگاه‌داده فقط از متغیر محیطی `OLIST_DB_PASSWORD` خوانده می‌شود. اگر تنظیم نشده باشد نوت‌بوک بلافاصله با پیام روشن متوقف می‌شود و چیزی نمی‌پرسد؛ پس Restart & Run All بدون دخالت دستی کار می‌کند.
3. اگر پایگاه‌داده یا نام کاربر شما با `../stage-1-kiarash-akbari/config/default.toml` فرق دارد، مسیر یک فایل تنظیمات محلی را در `OLIST_CONFIG` بگذارید.
4. همهٔ آزمایش‌هایی که ساختار را تغییر می‌دهند (حذف موقت Index، ساخت Index ردشده، درج انبوه) داخل تراکنشی اجرا می‌شوند که در پایان rollback می‌شود؛ پایگاه‌داده بعد از نوت‌بوک همان است که قبلش بود.

In [1]:
SEED = 42  # one seed for the whole notebook (nothing here samples, kept per the delivery guide)
import random
random.seed(SEED)

import json
import time
from pathlib import Path

import pandas as pd
import sqlalchemy as sa
from IPython.display import Markdown, display

from olist_perf import bench, bridge, db, evidence

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
started = time.perf_counter()

STAGE2 = Path.cwd()
if not (STAGE2 / "sql" / "queries").is_dir():
    raise RuntimeError("Run the notebook from the stage-2-kiarash-akbari folder.")
REPEATS, WARMUPS = 15, 3

engine = db.make_engine()          # password from the environment only; fails fast
print("database:", engine.url.database, "| session settings:", db.SESSION_SETTINGS)

database: dotin_s2 | session settings: {'jit': 'off', 'max_parallel_workers_per_gather': '0', 'work_mem': '32MB', 'statement_timeout': '120s'}


## ۱. وضعیت پایگاه‌داده

شمارش ردیف‌ها، viewها و فایل‌های SQL اعمال‌شده از خود پایگاه‌داده خوانده می‌شوند، نه از اعداد ثابت. فهرست migrationها نشان می‌دهد ساختار از فایل‌های نسخه‌دار `sql/*.sql` ساخته شده است (`004_performance_indexes.sql` همراه با 001 تا 003 و پیش از بارگذاری داده اعمال می‌شود).

In [2]:
with engine.connect() as connection:
    counts = pd.DataFrame([
        {"table": name, "rows": connection.scalar(sa.select(sa.func.count()).select_from(
            sa.table(name, schema="olist")))}
        for name in bridge.CSV
    ])
    views = pd.read_sql_query(sa.text(
        "SELECT viewname AS view FROM pg_views WHERE schemaname = :s "
        "UNION ALL SELECT matviewname || ' (materialized)' FROM pg_matviews "
        "WHERE schemaname = :s ORDER BY 1"), connection, params={"s": "olist"})
    applied = pd.read_sql_query(sa.text(
        "SELECT migration_name, applied_at FROM public.data_factory_migrations "
        "ORDER BY migration_name"), connection)
display(counts)
display(views)
display(applied)

,table,rows
0,customers,99441
1,sellers,3095
2,products,32951
3,orders,99441
4,order_items,112650
5,payments,103886
6,order_reviews,99224
7,geolocation,738332
8,product_category_name_translation,71


,view
0,mv_q1_monthly_category_sales (materialized)
1,mv_q3_delay_satisfaction (materialized)
2,mv_q4_seller_dashboard (materialized)
3,mv_q5_payment_distribution (materialized)
4,v_delivered_orders
5,v_order_item_sales


,migration_name,applied_at
0,001_create_olist_schema.sql,2026-11-25 09:51:03.500915+00:00
1,002_analytical_views.sql,2026-11-25 09:51:03.500915+00:00
2,003_functions_and_snapshots.sql,2026-11-25 09:51:03.500915+00:00
3,004_performance_indexes.sql,2026-11-25 09:51:03.500915+00:00
4,005_q4_seller_snapshot.sql,2026-11-25 09:51:57.607913+00:00
5,006_q5_payment_snapshot.sql,2026-11-25 09:51:57.607913+00:00
6,007_q1_monthly_category_snapshot.sql,2026-11-25 09:51:57.607913+00:00
7,008_q3_delay_satisfaction_snapshot.sql,2026-11-25 09:51:57.607913+00:00


## ۲. یکپارچگی ارجاعی دادهٔ خام (بخش ۱ صورت مسئله)

برای هر ارجاع، کلیدهای فرزند در CSV خام با کلیدهای والد مقایسه می‌شوند (فقط ستون‌های کلید خوانده می‌شوند). سه ارجاع آخر در schema کلید خارجی ندارند و دلیلش در `sql/001_create_olist_schema.sql` و `docs/schema_decisions.md` آمده است: نام دستهٔ بدون ترجمه و پیشوند کدپستی بدون مختصات، دادهٔ واقعی‌اند و حذف آن‌ها فروش واقعی را پاک می‌کرد. جدول دوم تعداد رکورد خام و بارگذاری‌شدهٔ هر جدول را مقایسه می‌کند؛ تنها اختلاف، ردیف‌های کاملاً تکراری geolocation است که loader پیش از درج حذف می‌کند.

In [3]:
fk = bridge.fk_audit(engine, bridge.csv_dir_from_config())
display(pd.DataFrame(fk["references"]))
display(pd.DataFrame(fk["row_counts"]))
evidence.save("fk-audit.json", fk);

,reference,fk_declared,child_rows,empty_key_rows,orphan_rows,distinct_orphan_keys
0,orders.customer_id -> customers.customer_id,True,99441,0,0,0
1,order_items.order_id -> orders.order_id,True,112650,0,0,0
2,order_items.product_id -> products.product_id,True,112650,0,0,0
3,order_items.seller_id -> sellers.seller_id,True,112650,0,0,0
4,payments.order_id -> orders.order_id,True,103886,0,0,0
5,order_reviews.order_id -> orders.order_id,True,99224,0,0,0
6,products.product_category_name -> product_cate...,False,32951,610,13,2
7,customers.customer_zip_code_prefix -> geolocat...,False,99441,0,278,157
8,sellers.seller_zip_code_prefix -> geolocation....,False,3095,0,7,7


,table,raw_csv_records,loaded_rows,not_loaded
0,customers,99441,99441,0
1,sellers,3095,3095,0
2,products,32951,32951,0
3,orders,99441,99441,0
4,order_items,112650,112650,0
5,payments,103886,103886,0
6,order_reviews,99224,99224,0
7,geolocation,1000163,738332,261831
8,product_category_name_translation,71,71,0


## ۳. نگهداری پس از بارگذاری و ساخت snapshot ماه‌های بسته

**VACUUM (ANALYZE):** بارگذاری انبوه، visibility map و آمار planner را خالی می‌گذارد تا autovacuum سراغ جدول‌ها بیاید. بدون آن، Index Only Scanها باید برای هر سطر به heap سر بزنند و planner با تخمین پیش‌فرض کار می‌کند؛ در اجرای آزمایشی این نوت‌بوک که بلافاصله بعد از بارگذاری و پیش از autovacuum شروع شد، Q2 بیش از سه برابر کندتر بود (`data/curated/evidence/benchmark-final-unvacuumed-run.json`). این سلول همان کار را صریح انجام می‌دهد تا اندازه‌گیری به زمان‌بندی autovacuum وابسته نباشد.

**snapshot:**
`olist.refresh_monthly_category_snapshot_range` (در 004) برای هر ماه، procedure یک‌ماههٔ 003 را صدا می‌زند؛ همه در یک تراکنش، پس خطا در هر ماه هیچ ماهی را نیمه‌کاره نمی‌گذارد. «ماه بسته» یعنی هر ماه پیش از ماهِ آخرین خرید. این همان کاری است که در عمل پس از هر بارگذاری یا هر شب زمان‌بندی می‌شود؛ Q1 ماه‌های بسته را از این جدول و ماه‌های بعد از آن را زنده حساب می‌کند.

In [4]:
snapshot_state = evidence.refresh_closed_months(engine)
display(pd.DataFrame(db.vacuum_analyze(engine)).set_index("table").T)
display(pd.Series(snapshot_state))

table,customers,sellers,product_category_name_translation,products,orders,order_items,payments,order_reviews,geolocation,monthly_category_snapshot
elapsed_ms,99.3,5.5,0.7,68.5,110.6,97.7,68.9,95.4,125.5,2.5


first_month                                2016-09-01
last_closed_month                          2018-09-01
refresh_ms                                      890.4
snapshot_rows                                    1282
last_month                                 2018-09-01
refreshed_at         2026-11-25 09:52:03.678303+00:00
dtype: object

## ۴. چرخهٔ بهینه‌سازی: پلن و زمان قبل و بعد برای هر پنج کوئری

**روش اندازه‌گیری (برای همهٔ کوئری‌ها یکسان):** یک session با `jit=off`، `work_mem=32MB` و `max_parallel_workers_per_gather=0` (ماشین اندازه‌گیری دو هستهٔ مشترک دارد و workerهای موازی زمان را به بار بقیه وابسته می‌کردند). سه اجرای گرم‌کردن، سپس پانزده اجرای اندازه‌گیری‌شده؛ زمان سمت کلاینت از ارسال تا دریافت آخرین ردیف است. حالت warm-cache است و تضمینی برای cold-cache یا بار هم‌زمان نیست. پلن با `EXPLAIN (ANALYZE, BUFFERS)` جدا گرفته می‌شود (زمان آن به‌خاطر سربار اندازه‌گیری هر گره کمی بیشتر است).

**قبل** = نسخهٔ اولیهٔ کوئری (`sql/baseline/`، کامیت 534c6ac) روی پایگاه‌داده‌ای که Indexهای 004 در یک تراکنش rollback‌شونده از آن حذف شده‌اند. **بعد** = `sql/queries/` با Indexها. خروجی دو نسخه باید ردیف‌به‌ردیف یکی باشد (SHA-256 پس از گرد کردن اعداد اعشاری شناور به ۱۲ رقم معنادار).

In [5]:
before_after = evidence.before_after(engine, STAGE2, repeats=REPEATS, warmups=WARMUPS)
summary = pd.DataFrame([
    {"query": name,
     "before_median_ms": r["before"]["median_ms"], "before_max_ms": r["before"]["max_ms"],
     "after_median_ms": r["after"]["median_ms"], "after_max_ms": r["after"]["max_ms"],
     "speedup": r["speedup"], "same_result": r["same_result"],
     "before_seq_scans": ", ".join(r["before"]["scans"]["seq_scan"]),
     "after_index_scans": ", ".join(r["after"]["scans"]["index_scan"])}
    for name, r in before_after.items()])
display(summary)
evidence.save("before-after.json", before_after)
assert summary["same_result"].all(), "an optimized query changed the answer"

,query,before_median_ms,before_max_ms,after_median_ms,after_max_ms,speedup,same_result,before_seq_scans,after_index_scans
0,q1_monthly_category_sales.sql,317.236,322.137,7.812,8.376,40.61,True,"order_items, orders, products","uq_monthly_category_snapshot_grain, uq_monthly..."
1,q2_delivery_by_state_month.sql,151.142,155.227,113.862,115.476,1.33,True,"orders, customers",
2,q3_delay_and_satisfaction.sql,201.740,204.139,160.395,161.544,1.26,True,"orders, order_reviews",ix_order_reviews_order_score
3,q4_top_sellers.sql,450.441,466.021,188.534,194.696,2.39,True,"order_items, orders, products, order_reviews","ix_orders_inactive, ix_orders_dispatch_cover, ..."
4,q5_payment_distribution.sql,460.547,468.844,155.806,159.201,2.96,True,"payments, orders","ix_payments_order_type_inst, ix_orders_inactive"


In [6]:
def show_plan(name):
    for side in ("before", "after"):
        nodes = pd.DataFrame(before_after[name][side]["plan_nodes"])
        nodes["node"] = ["  " * d + n for d, n in zip(nodes["depth"], nodes["node"])]
        display(Markdown(f"**{name} — {side}** (server execution "
                         f"{before_after[name][side]['server_execution_ms']:.1f} ms)"))
        display(nodes.drop(columns=["depth"]))

### Q1 — سبد و درآمد ماهانه به تفکیک دسته

**خواندن پلن قبل:** سه Seq Scan (order_items، orders، products) و دو Hash Join، بعد یک Sort روی همهٔ ~۱۱۲ هزار سطر آیتم برای `COUNT(DISTINCT order_id)` در GroupAggregate. گران‌ترین گره همین Sort و تجمیع است و هیچ فیلتری نیست که Index بتواند سطرها را کم کند: کوئری ذاتاً همهٔ تاریخ را می‌خواند. (این کوئری مورد بخش ۵ است؛ بخش ۸ را ببینید.)

**تغییر:** ماه‌های بسته از `olist.monthly_category_snapshot` خوانده می‌شوند و فقط ماه‌های بعد از آخرین ماه snapshot زنده تجمیع می‌شوند (`ix_orders_purchase_ts` بازهٔ زمانی را با Bitmap Index Scan می‌خواند).

**چرا کار کرد:** کاری که در هر باز شدن داشبورد تکرار می‌شد (تجمیع دو سال) یک بار در procedure انجام می‌شود؛ پلن بعد فقط حدود ۱۳۰۰ سطر آماده را به ترتیب Index یکتای snapshot می‌خواند و دُم زنده تقریباً خالی است.

In [7]:
show_plan("q1_monthly_category_sales.sql")

**q1_monthly_category_sales.sql — before** (server execution 362.9 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Aggregate,NaN,None,111219,1282,1,23519.49,361.477
1,Sort,NaN,None,111219,112101,1,19626.82,330.709
2,Hash Join,NaN,None,111219,112101,1,10026.93,163.272
3,Hash Join,NaN,None,111219,112101,1,8000.45,101.598
4,Seq Scan,order_items,None,112650,112650,1,3425.50,9.280
5,Hash,NaN,None,98178,98207,1,3052.01,35.165
6,Seq Scan,orders,None,98178,98207,1,3052.01,18.455
7,Hash,NaN,None,32951,32951,1,766.51,7.160
8,Seq Scan,products,None,32951,32951,1,766.51,3.083


**q1_monthly_category_sales.sql — after** (server execution 7.9 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Merge Append,NaN,NaN,38355,1282,1,13270.16,7.770
1,Index Scan,monthly_category_snapshot,uq_monthly_category_snapshot_grain,1282,1282,1,91.88,0.282
2,Aggregate,NaN,NaN,37073,0,1,12794.72,7.381
3,Result,NaN,NaN,1,1,1,0.34,0.022
4,Limit,NaN,NaN,1,1,1,0.32,0.018
5,Index Only Scan,monthly_category_snapshot,uq_monthly_category_snapshot_grain,1282,1,1,58.71,0.016
6,Sort,NaN,NaN,37073,0,1,11496.83,7.379
7,Hash Join,NaN,NaN,37073,0,1,8590.66,7.375
8,Hash Join,NaN,NaN,37073,0,1,7129.57,0.075
9,Seq Scan,order_items,NaN,112650,1,1,3425.50,0.008


### Q2 — نسبت تحویل دیرتر از وعده به تفکیک ایالت و ماه

**خواندن پلن قبل:** Seq Scan روی orders و customers و Hash Join؛ ~۹۶ هزار سفارش تحویل‌شده (حدود ۹۷٪ جدول) لازم است، پس Seq Scan انتخاب درستی است. تخمین و واقعیت سطرها نزدیک‌اند.

**تغییر:** (پیش‌تر در همین پروژه) تجمیع در یک CTE از نوع `MATERIALIZED` پیش از مرتب‌سازی نهایی، تا Sort روی ~۵۵۰ گروه انجام شود نه روی همهٔ سطرها. Index جدیدی برای Q2 ساخته نشد؛ Index پوشای جزئی امتحان شد و چون planner آن را انتخاب نکرد حذف شد (بخش ۷).

**چرا کار کرد:** مرتب‌سازی از سطح سفارش به سطح گروه منتقل شد؛ اسکن‌ها همان‌اند.

In [8]:
show_plan("q2_delivery_by_state_month.sql")

**q2_delivery_by_state_month.sql — before** (server execution 181.7 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Aggregate,NaN,None,93802,556,1,20360.59,180.599
1,Sort,NaN,None,93802,96470,1,15435.98,153.292
2,Hash Join,NaN,None,93802,96470,1,7454.69,102.860
3,Seq Scan,orders,None,93802,96470,1,3052.01,15.992
4,Hash,NaN,None,99441,99441,1,2444.41,28.383
5,Seq Scan,customers,None,99441,99441,1,2444.41,11.319


**q2_delivery_by_state_month.sql — after** (server execution 133.6 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Sort,NaN,None,93802,556,1,22236.63,133.510
1,Aggregate,NaN,None,93802,556,1,12379.29,133.164
2,Hash Join,NaN,None,93802,96470,1,7454.69,100.441
3,Seq Scan,orders,None,93802,96470,1,3052.01,16.411
4,Hash,NaN,None,99441,99441,1,2444.41,26.478
5,Seq Scan,customers,None,99441,99441,1,2444.41,10.975
6,CTE Scan,NaN,None,93802,556,1,1876.04,133.281


### Q3 — رابطهٔ تأخیر تحویل و امتیاز نظر

**خواندن پلن قبل:** Seq Scan روی order_reviews، HashAggregate برای میانگین امتیاز هر سفارش (عددی `numeric`) و Hash Join با سفارش‌های تحویل‌شده. بیشترین زمان در تجمیع هر سفارش و ساخت جدول hash است.

**تغییر:** `ix_order_reviews_order_score (order_id) INCLUDE (review_score)` و `ORDER BY order_id` در زیرکوئری نظرها.

**چرا کار کرد:** نظرها به ترتیب `order_id` از Index Only Scan می‌آیند (بدون دسترسی به heap و بدون Sort)، پس GroupAggregate جریانی کار می‌کند و جدول hash بزرگ میانی ساخته نمی‌شود. بهبود کوچک است و کوئری نزدیک بودجه می‌ماند؛ کار باقی‌مانده خواندن همهٔ سفارش‌های تحویل‌شده است.

In [9]:
show_plan("q3_delay_and_satisfaction.sql")

**q3_delay_and_satisfaction.sql — before** (server execution 234.9 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Aggregate,NaN,None,90669,4,1,29810.54,234.697
1,Sort,NaN,None,92746,95824,1,20939.16,218.664
2,Hash Join,NaN,None,92746,95824,1,13055.29,200.857
3,Seq Scan,orders,None,93802,96470,1,3052.01,15.634
4,Hash,NaN,None,98321,98673,1,5745.64,132.006
5,Subquery Scan,NaN,None,98321,98673,1,5745.64,109.657
6,Aggregate,NaN,None,98321,98673,1,4762.43,100.683
7,Seq Scan,order_reviews,None,99224,99224,1,2789.24,8.233


**q3_delay_and_satisfaction.sql — after** (server execution 192.9 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Aggregate,NaN,NaN,90669,4,1,32381.46,192.812
1,Sort,NaN,NaN,92746,95824,1,23510.07,176.892
2,Hash Join,NaN,NaN,92746,95824,1,15626.20,157.698
3,Aggregate,NaN,NaN,98321,98673,1,7377.97,68.781
4,Index Only Scan,order_reviews,ix_order_reviews_order_score,99224,99224,1,5404.78,11.924
5,Hash,NaN,NaN,93802,96470,1,3052.01,32.959
6,Seq Scan,orders,NaN,93802,96470,1,3052.01,16.003


### Q4 — ۲۰ فروشندهٔ برتر

**خواندن پلن قبل:** چهار Seq Scan و تجمیع همهٔ فروشنده‌ها (حدود ۳ هزار) با جزئیات آماده‌سازی و امتیاز، در حالی که فقط ۲۰ ردیف لازم است. تخمین سطرهای reviews پس از تجمیع بسیار کمتر از واقعیت است.

**تغییر:** اول فقط درآمد هر فروشنده (با `NOT EXISTS` روی Index جزئی کوچک `ix_orders_inactive` به‌جای Join با همهٔ سفارش‌ها) و انتخاب ۲۰ نفر؛ بعد فقط سطرهای همین ۲۰ نفر (Hash Join با order_items)، تاریخ‌های هر سفارش از `ix_orders_dispatch_cover` (Index Only Scan) و امتیاز هر سفارش از `ix_order_reviews_order_score`.

**چرا کار کرد:** جزئیات گران فقط برای حدود یک‌پنجم سفارش‌ها حساب می‌شود و دسترسی به سفارش‌ها و نظرها با کلید و بدون heap است. این کوئری بیشترین نزدیکی را به بودجه دارد.

In [10]:
show_plan("q4_top_sellers.sql")

**q4_top_sellers.sql — before** (server execution 521.8 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Limit,NaN,None,20,20,1,1661197.69,521.438
1,Sort,NaN,None,200,20,1,1661198.14,521.433
2,Aggregate,NaN,None,200,3053,1,1661192.32,520.882
3,Hash Join,NaN,None,54675816,99542,1,20914.34,458.398
4,Aggregate,NaN,None,111219,99542,1,12529.36,266.799
5,Hash Join,NaN,None,111219,112101,1,9470.84,157.089
6,Hash Join,NaN,None,111219,112101,1,8000.45,105.454
7,Seq Scan,order_items,None,112650,112650,1,3425.50,9.325
8,Hash,NaN,None,98178,98207,1,3052.01,35.756
9,Seq Scan,orders,None,98178,98207,1,3052.01,17.382


**q4_top_sellers.sql — after** (server execution 229.1 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Aggregate,NaN,NaN,20,20,1,10623.33,229.014
1,Limit,NaN,NaN,20,20,1,5844.90,63.089
2,Sort,NaN,NaN,2370,20,1,5850.77,63.084
3,Aggregate,NaN,NaN,2370,3053,1,5781.78,62.650
4,Hash Join,NaN,NaN,111219,112101,1,4918.02,31.018
5,Seq Scan,order_items,NaN,112650,112650,1,3425.50,8.704
6,Hash,NaN,NaN,1263,1234,1,67.22,0.271
7,Index Only Scan,orders,ix_orders_inactive,1263,1234,1,67.22,0.131
8,Sort,NaN,NaN,94,18908,1,4775.31,220.433
9,Nested Loop,NaN,NaN,94,18908,1,4772.00,212.299


### Q5 — توزیع روش پرداخت و اقساط و رابطه با ارزش سفارش

**خواندن پلن قبل:** CTE مجموع هر سفارش دو بار خوانده می‌شود، Hash Join با ~۹۸ هزار سفارش و یک Sort بزرگ برای `COUNT(DISTINCT order_id)` در هر گروه.

**تغییر:** یک WindowAgg روی `ix_payments_order_type_inst` که از قبل به ترتیب (سفارش، روش، اقساط) است: مجموع هر سفارش، اولین سطر هر سفارش (برای همبستگی) و اولین سطر هر گروه در هر سفارش (برای شمارش یکتا) در یک گذر؛ سپس یک `GROUPING SETS` برای ۲۷ گروه و سطر کل.

**چرا کار کرد:** دادهٔ پرداخت یک بار و بدون Sort خوانده می‌شود و Join با orders به یک anti-join کوچک تبدیل شده است. همبستگی از نوع double precision است؛ ترتیب جمع زدن رقم‌های آخر آن را جابه‌جا می‌کند، به همین دلیل مقایسه با ۱۲ رقم معنادار است.

In [11]:
show_plan("q5_payment_distribution.sql")

**q5_payment_distribution.sql — before** (server execution 538.5 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,WindowAgg,NaN,None,10257,27,1,26323.91,537.325
1,Hash Join,NaN,None,102567,102573,1,6771.81,95.596
2,Seq Scan,payments,None,103886,103886,1,2219.86,8.056
3,Hash,NaN,None,98178,98207,1,3052.01,32.038
4,Seq Scan,orders,None,98178,98207,1,3052.01,15.415
5,Aggregate,NaN,None,200,98206,1,2823.09,165.332
6,CTE Scan,NaN,None,102567,102573,1,2051.34,85.696
7,Nested Loop,NaN,None,10257,27,1,16549.51,537.268
8,Aggregate,NaN,None,10257,27,1,16415.28,511.250
9,Sort,NaN,None,102567,102573,1,14697.28,486.624


**q5_payment_distribution.sql — after** (server execution 179.4 ms)

,node,relation,index,est_rows,actual_rows,loops,total_cost,actual_total_ms
0,Sort,NaN,NaN,1,27,1,17773.88,179.373
1,Aggregate,NaN,NaN,106,28,1,17769.08,179.277
2,WindowAgg,NaN,NaN,102567,102573,1,11099.84,123.657
3,Merge Join,NaN,NaN,102567,102573,1,7509.99,31.324
4,Index Only Scan,payments,ix_payments_order_type_inst,103886,103886,1,7166.71,14.627
5,Index Only Scan,orders,ix_orders_inactive,1263,1234,1,67.22,0.167
6,Nested Loop,NaN,NaN,1,27,1,4.79,179.354
7,CTE Scan,NaN,NaN,1,27,1,2.38,179.264
8,CTE Scan,NaN,NaN,1,1,27,2.38,0.003


## ۵. بنچمارک نهایی پنج کوئری داشبورد

همان روش بخش ۴، این بار فقط روی `sql/queries/`. معیار پذیرش: میانه و بیشینهٔ هر پانزده اجرا زیر ۲۰۰ میلی‌ثانیه.

In [12]:
final = bench.benchmark_dir(engine, STAGE2 / "sql" / "queries", repeats=REPEATS, warmups=WARMUPS)
final_table = pd.DataFrame([
    {"query": name, "median_ms": r["median_ms"], "p95_ms": r["p95_ms"], "max_ms": r["max_ms"],
     "server_execution_ms": round(r["server_execution_ms"], 1), "rows": r["rows"],
     "all_runs_under_200ms": r["all_runs_under_budget"]}
    for name, r in final["queries"].items()])
display(final_table)
evidence.save("benchmark-final.json", {
    **{k: v for k, v in final.items() if k != "queries"},
    "queries": {n: {k: v for k, v in r.items() if k != "plan"}
                for n, r in final["queries"].items()}})
print("all five under budget on every run:", bool(final_table["all_runs_under_200ms"].all()))

,query,median_ms,p95_ms,max_ms,server_execution_ms,rows,all_runs_under_200ms
0,q1_monthly_category_sales.sql,7.850,8.965,8.965,7.7,1282,True
1,q2_delivery_by_state_month.sql,112.744,115.812,115.812,134.0,556,True
2,q3_delay_and_satisfaction.sql,155.568,160.298,160.298,189.8,4,True
3,q4_top_sellers.sql,187.516,188.134,188.134,228.0,20,True
4,q5_payment_distribution.sql,155.575,158.497,158.497,176.5,27,True


all five under budget on every run: True


## ۶. شاهد هر Index

برای هر Index فایل 004، کوئری‌ای که به خاطرش ساخته شده دو بار اندازه‌گیری می‌شود: با Index، و در تراکنشی که Index را حذف و سپس rollback می‌کند. ستون `used_in_plan` از خود پلن می‌آید. برای `ix_orders_purchase_ts` snapshot موقتاً تا 2018-06 کوتاه می‌شود (باز هم داخل همان تراکنش) تا دُم زندهٔ Q1 سه ماه واقعی داشته باشد؛ با snapshot کامل این دُم تقریباً خالی است.

In [13]:
index_ev = evidence.index_evidence(engine, STAGE2)
display(pd.DataFrame([
    {"index": name, "table": r["table"], "query": r["query"], "used_in_plan": r["used_in_plan"],
     "median_with_ms": r["median_with_ms"], "median_without_ms": r["median_without_ms"],
     "saved_ms": r["saved_ms"], "same_result": r["same_result"],
     "scans_without": ", ".join(r["without_index"]["scans"]["index_scan"]
                                + r["without_index"]["scans"]["seq_scan"])}
    for name, r in index_ev.items()]))

,index,table,query,used_in_plan,median_with_ms,median_without_ms,saved_ms,same_result,scans_without
0,ix_orders_purchase_ts,orders,q1_monthly_category_sales.sql,True,57.035,64.401,7.366,True,"uq_monthly_category_snapshot_grain, uq_monthly..."
1,ix_orders_inactive,orders,q5_payment_distribution.sql,True,155.815,165.537,9.722,True,"ix_payments_order_type_inst, orders"
2,ix_orders_dispatch_cover,orders,q4_top_sellers.sql,True,186.301,203.886,17.585,True,"ix_orders_inactive, pk_orders, ix_order_review..."
3,ix_order_reviews_order_score,order_reviews,q3_delay_and_satisfaction.sql,True,156.830,225.406,68.576,True,"order_reviews, orders"
4,ix_payments_order_type_inst,payments,q5_payment_distribution.sql,True,154.644,233.613,78.969,True,"ix_orders_inactive, payments"


## ۷. هزینهٔ پنهان Index

**حجم:** اندازهٔ هر Index و نسبت مجموع Indexهای 004 به حجم heap همان جدول‌ها؛ و همان Indexها پس از `REINDEX` (در تراکنش rollback‌شونده)، چون 004 پیش از بارگذاری اجرا می‌شود و Index در حین درج‌ها با شکستن صفحه‌ها بزرگ‌تر از ساخت یک‌باره می‌شود. **نوشتن:** هر جدول یک بار در یک کپیِ دارای Indexهای 004 و یک بار در کپیِ فقط با کلید اصلی درج می‌شود (`INSERT ... SELECT` همهٔ سطرها، سه تکرار، در schema موقتی که rollback می‌شود؛ کلید خارجی در هیچ‌کدام از دو حالت نیست). **Indexهای حذف‌شده:** Index پوشای جزئی برای Q2 و Index `(seller_id, order_id)` برای Q4 ساخته، اندازه‌گیری و حذف شدند (هر کدام در تراکنشی که rollback می‌شود دوباره ساخته و اندازه‌گیری می‌شود).

In [14]:
sizes = evidence.index_sizes(engine)
size_sum = evidence.size_summary(sizes)
display(pd.DataFrame(sizes))
display(pd.DataFrame(size_sum["tables"]).T)
print("004 indexes:", size_sum["total_004_bytes"], "bytes =",
      size_sum["ratio"], "x the heap of the tables they index")
rebuilt = evidence.rebuilt_sizes(engine)   # REINDEX in a rolled-back transaction
display(pd.DataFrame(rebuilt))

,table_name,index_name,index_bytes,table_bytes,from_004
0,customers,pk_customers,7856128,11878400,False
1,geolocation,pk_geolocation,16605184,58286080,False
2,monthly_category_snapshot,uq_monthly_category_snapshot_grain,73728,122880,False
3,order_items,pk_order_items,6668288,18833408,False
4,order_reviews,ix_order_reviews_order_score,8019968,14721024,True
5,order_reviews,pk_order_reviews,12197888,14721024,False
6,orders,ix_orders_dispatch_cover,10936320,14819328,True
7,orders,ix_orders_inactive,98304,14819328,True
8,orders,ix_orders_purchase_ts,2662400,14819328,True
9,orders,pk_orders,7864320,14819328,False


,table_bytes,index_004_bytes,all_index_bytes,ratio_004_to_table
order_reviews,14721024.0,8019968.0,20217856.0,0.5448
orders,14819328.0,13697024.0,21561344.0,0.9243
payments,9674752.0,11485184.0,19578880.0,1.1871


004 indexes: 33202176 bytes = 0.8467 x the heap of the tables they index


,index,bytes_as_loaded,bytes_after_reindex,ratio
0,ix_orders_purchase_ts,2662400,2244608,1.186
1,ix_orders_inactive,98304,90112,1.091
2,ix_orders_dispatch_cover,10936320,8585216,1.274
3,ix_order_reviews_order_score,8019968,5881856,1.364
4,ix_payments_order_type_inst,11485184,8732672,1.315


In [15]:
writes = evidence.write_cost(engine, repeats=3)
display(pd.DataFrame(writes).T[["rows_inserted", "indexes_004", "median_without_ms",
                                "median_with_ms", "slowdown_ratio"]])
rejected = evidence.rejected_indexes(engine, STAGE2)
display(pd.DataFrame([{"index": name, **{k: r[k] for k in (
    "query", "used_in_plan", "index_bytes", "build_ms", "median_without_ms",
    "median_with_ms", "same_result")}} for name, r in rejected.items()]))
evidence.save("index-evidence.json", {"indexes": index_ev, "rejected_indexes": rejected})
evidence.save("index-cost.json", {"sizes": sizes, "summary": size_sum, "write_cost": writes,
                                  "rebuilt_sizes": rebuilt});

,rows_inserted,indexes_004,median_without_ms,median_with_ms,slowdown_ratio
orders,99441,"[ix_orders_purchase_ts, ix_orders_inactive, ix...",629.1,1140.6,1.813
order_items,112650,[],1186.1,1173.1,0.989
order_reviews,99224,[ix_order_reviews_order_score],947.5,1223.4,1.291
payments,103886,[ix_payments_order_type_inst],657.8,984.4,1.496


,index,query,used_in_plan,index_bytes,build_ms,median_without_ms,median_with_ms,same_result
0,ix_orders_delivered_partial,q2_delivery_by_state_month.sql,False,8331264,177.037,113.378,114.132,True
1,ix_order_items_seller_order,q4_top_sellers.sql,True,9797632,202.994,188.655,189.550,True


## ۸. کوئری‌ای که Index نجاتش نمی‌دهد: Q1 (بخش ۵ صورت مسئله)

پلن «قبل» Q1 در بخش ۴ نشان می‌دهد هزینه در Sort و GroupAggregate روی همهٔ سطرهای آیتم است، نه در پیدا کردن سطرها. صورت سؤال (همهٔ ماه‌ها × همهٔ دسته‌ها، با سفارش یکتا) هیچ فیلتری ندارد که Index بتواند سطرهای خوانده‌شده را کم کند، و `COUNT(DISTINCT)` روی گروه‌ها مرتب‌سازی یا hash روی کل داده را اجباری می‌کند. حتی با همهٔ Indexهای 004 روی پایگاه‌داده، planner برای نسخهٔ اولیه همان Seq Scanها را انتخاب می‌کند (ستون `before` جدول بخش ۴ با Indexها حذف‌شده است؛ جدول زیر نسخهٔ اولیه را با Indexها هم نشان می‌دهد).

راه‌حل غیر Index: پیش‌محاسبهٔ ماه‌های بسته با procedure و محاسبهٔ زندهٔ فقط ماه‌های باز. **بهایی که پرداخت شد:** تازگی ماه‌های بسته به آخرین اجرای refresh بسته است، یک جدول و یک job نگهداری اضافه شد، و اگر job اجرا نشود دُم زنده بزرگ و کوئری کندتر می‌شود (جدول دوم: زمان Q1 بر حسب تعداد ماه‌هایی که snapshot پوشش می‌دهد؛ با snapshot خالی، پاسخ همچنان درست ولی کند است).

In [16]:
with db.session(engine) as connection:
    baseline_with_indexes = bench.measure(
        connection, bench.read_query(STAGE2 / "sql" / "baseline" / "q1_monthly_category_sales.sql"),
        repeats=9, warmups=2)
display(pd.Series({"baseline q1 with all 004 indexes, median_ms": baseline_with_indexes["median_ms"],
                   "index scans in its plan": baseline_with_indexes["scans"]["index_scan"],
                   "seq scans in its plan": baseline_with_indexes["scans"]["seq_scan"]}))
tail = evidence.q1_tail_series(engine, STAGE2)
display(pd.DataFrame(tail).drop(columns=["scans"]))

baseline q1 with all 004 indexes, median_ms                             313.34
index scans in its plan                                                     []
seq scans in its plan                          [order_items, orders, products]
dtype: object

,snapshot_through,median_ms,max_ms,result_sha256
0,2018-09-01,7.533,8.329,b229b266de00f7211138d19e9398b44e811e225b1699c6...
1,2018-08-01,18.206,75.180,b229b266de00f7211138d19e9398b44e811e225b1699c6...
2,2018-06-01,58.688,64.796,b229b266de00f7211138d19e9398b44e811e225b1699c6...
3,2018-03-01,126.366,130.014,b229b266de00f7211138d19e9398b44e811e225b1699c6...
4,2017-12-01,189.596,198.144,b229b266de00f7211138d19e9398b44e811e225b1699c6...
5,empty snapshot,318.127,322.299,b229b266de00f7211138d19e9398b44e811e225b1699c6...


## ۹. View یا جدول snapshot برای سنگین‌ترین تحلیل (Q1)

چهار راه برای یک پاسخ: view زنده (`v_order_item_sales`)، فقط جدول snapshot، materialized view فایل 007، و ترکیب انتخاب‌شده. هر چهار باید همان SHA-256 را بدهند. آزمون تازگی: در یک تراکنش rollback‌شونده یک سفارش جدید (۱۰۰ واحد) یک بار در آخرین ماه بسته و یک بار در ماه باز درج می‌شود و بررسی می‌شود کدام روش آن را می‌بیند.

In [17]:
vs = evidence.view_vs_snapshot(engine, STAGE2, repeats=REPEATS, warmups=WARMUPS)
display(pd.DataFrame([{"variant": k, "median_ms": v["median_ms"], "max_ms": v["max_ms"],
                       "rows": v["rows"], "result_sha256": v["result_sha256"][:16]}
                      for k, v in vs["timings"].items()]))
display(pd.DataFrame(vs["space"]))
print("refresh of all closed months (ms, 3 runs, rolled back):",
      vs["snapshot_refresh_all_closed_months_ms"])
display(pd.DataFrame({k: v["sees_new_order"] for k, v in vs["freshness"].items()}))
evidence.save("view-vs-snapshot.json", {**vs, "q1_tail_series": tail,
                                        "baseline_q1_with_indexes": {
                                            k: v for k, v in baseline_with_indexes.items()
                                            if k != "plan"}});

,variant,median_ms,max_ms,rows,result_sha256
0,live_view (sql/baseline/q1 over v_order_item_s...,326.547,394.485,1282,b229b266de00f721
1,snapshot_table_only,2.149,2.650,1282,b229b266de00f721
2,materialized_view mv_q1 (007),1.997,2.485,1282,b229b266de00f721
3,"hybrid (sql/queries/q1, chosen)",7.788,9.074,1282,b229b266de00f721


,relation,total_bytes
0,olist.monthly_category_snapshot,237568
1,olist.mv_q1_monthly_category_sales,221184


refresh of all closed months (ms, 3 runs, rolled back): [824.1, 838.0, 840.5]


,order_in_a_closed_month,order_in_the_live_tail_month
live_view (sql/baseline/q1 over v_order_item_sales),True,True
snapshot_table_only,False,False
materialized_view mv_q1 (007),False,False
"hybrid (sql/queries/q1, chosen)",False,True


## ۱۰. Procedure و Function

`olist.delivery_delay_days` (003) یک Function از نوع `IMMUTABLE` است و داخل `SELECT`، `WHERE` و `CASE` کوئری‌های Q2 و پایه‌ٔ Q3 به کار می‌رود؛ یک مقدار برمی‌گرداند و نمی‌تواند تراکنش را مدیریت کند یا جدول را تغییر دهد (`IMMUTABLE`). `olist.refresh_monthly_category_snapshot` (003) و نسخهٔ بازه‌ای آن (004) Procedure هستند: با `CALL` اجرا می‌شوند، داخل `SELECT` قابل استفاده نیستند، و داده را حذف و درج می‌کنند. دو اجرای پشت‌سرهم برای یک ماه باید همان سطرها را بدهد (idempotent)؛ رفتار در خطای میانهٔ کار در `tests/test_postgres_integration.py` آزمون شده است.

In [18]:
month = {"month": "2018-05-01"}
grain = sa.text("SELECT count(*) AS rows, sum(sales_value) AS sales FROM "
                "olist.monthly_category_snapshot WHERE month_start = CAST(:month AS date)")
call = sa.text("CALL olist.refresh_monthly_category_snapshot(CAST(:month AS date))")
with engine.begin() as connection:
    connection.execute(call, month)
    first = connection.execute(grain, month).mappings().one()
    connection.execute(call, month)
    second = connection.execute(grain, month).mappings().one()
display(pd.DataFrame([dict(first), dict(second)], index=["first CALL", "second CALL"]))
assert dict(first) == dict(second)

with engine.connect() as connection:
    delays = pd.read_sql_query(sa.text(
        "SELECT olist.delivery_delay_days(order_delivered_customer_date, "
        "order_estimated_delivery_date) > 0 AS late, count(*) AS orders "
        "FROM olist.v_delivered_orders GROUP BY 1 ORDER BY 1"), connection)
display(delays)

,rows,sales
first CALL,65,1145686.46
second CALL,65,1145686.46


,late,orders
0,False,89936
1,True,6534


## ۱۱. پل Python و SQL

- **خواندن دسته‌ای از View:** ~۱۱۲ هزار سطر `v_order_item_sales` با `stream_results` و `chunksize` خوانده و دسته‌به‌دسته تجمیع می‌شود؛ هیچ‌وقت کل نتیجه در حافظهٔ کلاینت نیست. مقدار `since` پارامتر bind است.
- **نوشتن با نوع ستون مشخص:** خروجی Q2 با `to_sql` و نگاشت صریح (`DATE`، `CHAR(2)`، `INTEGER`، `NUMERIC(6,4)`) در schema `reporting` نوشته و نوع ذخیره‌شده از `information_schema` خوانده می‌شود.
- هیچ کوئری با چسباندن رشته ساخته نمی‌شود؛ شناسه‌ها با `psycopg.sql.Identifier` و مقدارها با پارامتر (آزمون `NoStringBuiltSqlTests`).

In [19]:
chunked = bridge.read_view_in_chunks(engine, since="2017-01-01", chunksize=20_000)
print({k: v for k, v in chunked.items() if k != "monthly"})
display(chunked["monthly"].tail(6))
with engine.connect() as connection:
    q2 = pd.read_sql_query(sa.text(bench.read_query(
        STAGE2 / "sql" / "queries" / "q2_delivery_by_state_month.sql")), connection)
stored_types = bridge.write_typed(engine, q2)
display(stored_types)
print("rows written:", stored_types.attrs["rows_written"])
evidence.save("python-bridge.json", {
    "chunked_read": {k: v for k, v in chunked.items() if k != "monthly"},
    "chunked_monthly_revenue": chunked["monthly"].to_dict(orient="records"),
    "to_sql_types": stored_types.to_dict(orient="records"),
    "to_sql_rows": stored_types.attrs["rows_written"]});

{'chunksize': 20000, 'chunks': 6, 'rows': 111753, 'largest_chunk_rows': 20000}


,month,revenue
15,2018-04,1156248.89
16,2018-05,1145686.46
17,2018-06,1020381.90
18,2018-07,1039783.58
19,2018-08,996973.51
20,2018-09,166.46


,column_name,data_type,character_maximum_length,numeric_precision,numeric_scale
0,purchase_month,date,NaN,NaN,NaN
1,customer_state,character,2.0,NaN,NaN
2,delivered_orders,integer,NaN,32.0,0.0
3,late_orders,integer,NaN,32.0,0.0
4,late_ratio,numeric,NaN,6.0,4.0


rows written: 556


## ۱۲. تکرارپذیری

`scripts/determinism_two_loads.py` پایگاه‌داده را دو بار از صفر با همان یک فرمان می‌سازد و محتوای همهٔ جدول‌ها و نتیجهٔ پنج کوئری را مقایسه می‌کند (این اسکریپت مخرب است و جزو نوت‌بوک اجرا نمی‌شود؛ نتیجه‌اش در `data/curated/evidence/determinism-two-loads.json` است). این سلول هش نتیجهٔ کوئری‌های همین اجرا را با آن فایل مقایسه می‌کند. زمان‌ها در هر اجرا کمی فرق می‌کنند؛ پاسخ‌ها نباید فرق کنند.

In [20]:
two_loads = json.loads((evidence.EVIDENCE_DIR / "determinism-two-loads.json").read_text(encoding="utf-8"))
print("two independent loads -> identical tables:", two_loads["identical_tables"],
      "| identical query results:", two_loads["identical_query_results"])
check = pd.DataFrame([
    {"query": name, "this_run": final["queries"][name]["result_sha256"][:16],
     "two_loads_file": two_loads["runs"][0]["queries"][name]["result_sha256"][:16]}
    for name in bench.QUERY_FILES])
check["same"] = check["this_run"] == check["two_loads_file"]
display(check)

two independent loads -> identical tables: True | identical query results: True


,query,this_run,two_loads_file,same
0,q1_monthly_category_sales.sql,b229b266de00f721,b229b266de00f721,True
1,q2_delivery_by_state_month.sql,0fb5ea786a0a323e,0fb5ea786a0a323e,True
2,q3_delay_and_satisfaction.sql,4081b798251ec41f,4081b798251ec41f,True
3,q4_top_sellers.sql,333ddaceed600eb5,333ddaceed600eb5,True
4,q5_payment_distribution.sql,d37ffa0db53f5a81,d37ffa0db53f5a81,True


## ۱۳. بستن اتصال‌ها و جمع‌بندی اجرا

همهٔ اتصال‌ها با `with` باز و بسته شده‌اند؛ در پایان pool موتور dispose می‌شود و از یک اتصال جداگانه بررسی می‌شود که هیچ session با نام برنامهٔ این نوت‌بوک (`olist_perf`) روی پایگاه‌داده باقی نمانده باشد.

In [21]:
engine.dispose()
leftover = db.leftover_sessions(engine)
elapsed = round(time.perf_counter() - started, 1)
evidence.save("notebook-run.json", {"elapsed_seconds": elapsed, "leftover_sessions": leftover,
                                    "seed": SEED, "repeats": REPEATS, "warmups": WARMUPS,
                                    "snapshot": snapshot_state})
print("leftover sessions:", leftover)
print("notebook elapsed seconds:", elapsed, "(cap: 600)")
assert leftover == []

leftover sessions: []
notebook elapsed seconds: 136.2 (cap: 600)


## AI usage

از دستیار هوش مصنوعی برای بازنویسی کوئری‌های Q1، Q3، Q4 و Q5، پیشنهاد و اندازه‌گیری Indexهای 004، نوشتن پکیج `olist_perf` و سازندهٔ این نوت‌بوک کمک گرفته شد. همهٔ اعداد از اجرای همین نوت‌بوک روی دادهٔ واقعی Olist آمده‌اند و در `data/curated/evidence/` ذخیره شده‌اند؛ روی ماشین خودتان با Restart & Run All دوباره تولید می‌شوند و ممکن است کمی فرق کنند.